In [0]:
# ===== 04_validation =====

from pyspark.sql.functions import col

silver_base = "/Volumes/dbacademy/default/ecommerce_project/silver"
gold_base = "/Volumes/dbacademy/default/ecommerce_project/gold"

validation_results = []

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    validation_results.append((name, status, detail))
    print(f"[{status}] {name} {detail}")

# --- 1. Row count sanity checks ---
df_orders = spark.read.format("delta").load(f"{silver_base}/orders")
df_customers = spark.read.format("delta").load(f"{silver_base}/customers")
df_items = spark.read.format("delta").load(f"{silver_base}/order_items")
df_payments = spark.read.format("delta").load(f"{silver_base}/order_payments")
df_reviews = spark.read.format("delta").load(f"{silver_base}/order_reviews")
df_rees46 = spark.read.format("delta").load(f"{silver_base}/rees46_events")

check("orders row count == customers row count", df_orders.count() == df_customers.count(),
      f"(orders={df_orders.count()}, customers={df_customers.count()})")

check("order_items row count >= orders row count", df_items.count() >= df_orders.count(),
      f"(items={df_items.count()}, orders={df_orders.count()})")

check("payments row count within expected range (post-cleaning)",
      100000 <= df_payments.count() <= 104000,
      f"(payments={df_payments.count()})")

check("rees46_events row count matches expected 7-day sample",
      df_rees46.count() == 8829315,
      f"(rees46={df_rees46.count()})")

# --- 2. Critical null checks ---
check("orders.order_id has zero nulls", df_orders.filter(col("order_id").isNull()).count() == 0)
check("customers.customer_id has zero nulls", df_customers.filter(col("customer_id").isNull()).count() == 0)
check("order_items.order_id has zero nulls", df_items.filter(col("order_id").isNull()).count() == 0)
check("order_reviews.review_score has zero nulls", df_reviews.filter(col("review_score").isNull()).count() == 0)
check("rees46_events.category_level_1 has zero nulls", df_rees46.filter(col("category_level_1").isNull()).count() == 0)

# --- 3. Duplicate checks ---
check("orders.order_id has no duplicates",
      df_orders.count() == df_orders.select("order_id").distinct().count())
check("customers.customer_id has no duplicates",
      df_customers.count() == df_customers.select("customer_id").distinct().count())

# --- 4. Referential integrity ---
orphan_items = df_items.join(df_orders, "order_id", "left_anti")
check("all order_items.order_id exist in orders", orphan_items.count() == 0,
      f"(orphan rows: {orphan_items.count()})")

orphan_reviews = df_reviews.join(df_orders, "order_id", "left_anti")
check("all order_reviews.order_id exist in orders", orphan_reviews.count() == 0,
      f"(orphan rows: {orphan_reviews.count()})")

# --- 5. Gold layer existence checks ---
gold_tables = ["category_comparison", "olist_monthly_sales", "olist_customer_ltv",
               "olist_seller_performance", "olist_review_trends", "rees46_overall_funnel"]

for table in gold_tables:
    try:
        df = spark.read.format("delta").load(f"{gold_base}/{table}")
        check(f"Gold table '{table}' is readable and non-empty", df.count() > 0, f"(rows={df.count()})")
    except Exception as e:
        check(f"Gold table '{table}' is readable and non-empty", False, f"(error: {e})")

# --- Summary ---
total = len(validation_results)
passed = sum(1 for _, status, _ in validation_results if status == "PASS")
print(f"\n=== Validation Summary: {passed}/{total} checks passed ===")

if passed < total:
    print("\nFailed checks:")
    for name, status, detail in validation_results:
        if status == "FAIL":
            print(f"  - {name} {detail}")

[PASS] orders row count == customers row count (orders=99441, customers=99441)
[PASS] order_items row count >= orders row count (items=112650, orders=99441)
[PASS] payments row count within expected range (post-cleaning) (payments=103883)
[PASS] rees46_events row count matches expected 7-day sample (rees46=8829315)
[PASS] orders.order_id has zero nulls 
[PASS] customers.customer_id has zero nulls 
[PASS] order_items.order_id has zero nulls 
[PASS] order_reviews.review_score has zero nulls 
[PASS] rees46_events.category_level_1 has zero nulls 
[PASS] orders.order_id has no duplicates 
[PASS] customers.customer_id has no duplicates 
[PASS] all order_items.order_id exist in orders (orphan rows: 0)
[PASS] all order_reviews.order_id exist in orders (orphan rows: 0)
[PASS] Gold table 'category_comparison' is readable and non-empty (rows=15)
[PASS] Gold table 'olist_monthly_sales' is readable and non-empty (rows=23)
[PASS] Gold table 'olist_customer_ltv' is readable and non-empty (rows=93358)